# 1. Load data


In [3]:
import pandas as pd

path = r"C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\data\processed\year7_125activities.csv"

df = pd.read_csv(path)

# 2. Select the three candidate variable groups
MEMS7GR_ALL is the summary column, excluded here


In [15]:
mems7gr_cols = [c for c in df.columns if c.startswith('MEMS7GR_') and c != 'MEMS7GR_ALL']
months12_cols = [c for c in df.columns if c.startswith('MONTHS_12_')]
days_cols = [c for c in df.columns if c.startswith('DAYS10P60GR_')]


# 3. Compute missing rate per activity


In [5]:
def missing_summary(cols, prefix):
    return pd.Series({c.replace(prefix, ''): df[c].isna().mean() for c in cols})

mems7gr_missing = missing_summary(mems7gr_cols, 'MEMS7GR_')
months12_missing = missing_summary(months12_cols, 'MONTHS_12_')
days_missing = missing_summary(days_cols, 'DAYS10P60GR_')


# 4. Combine into one comparison table
Activities with the highest missing rate
Count activities with zero missing vs high missing

In [6]:
missing_table = pd.DataFrame({
    'MEMS7GR': mems7gr_missing,
    'MONTHS_12': months12_missing,
    'DAYS10P60GR': days_missing
})

print('Overall distribution')
print(missing_table.describe())

print(missing_table.sort_values('MEMS7GR', ascending=False).head(10))

print('Activities with zero missing across all three:', (missing_table == 0).all(axis=1).sum())
print('Activities with MEMS7GR missing rate above 20%:', (missing_table['MEMS7GR'] > 0.2).sum())


Overall distribution
          MEMS7GR   MONTHS_12  DAYS10P60GR
count  125.000000  125.000000   125.000000
mean     0.103510    0.103510     0.103510
std      0.119167    0.119167     0.119167
min      0.000000    0.000000     0.000000
25%      0.000000    0.000000     0.000000
50%      0.000000    0.000000     0.000000
75%      0.239606    0.239606     0.239606
max      0.239606    0.239606     0.239606
                     MEMS7GR  MONTHS_12  DAYS10P60GR
MARTIALCHINESE_S05  0.239606   0.239606     0.239606
AIRGUN_S08          0.239606   0.239606     0.239606
MARTIALOTHER_S06    0.239606   0.239606     0.239606
RIFLE_S09           0.239606   0.239606     0.239606
SKIING_T01          0.239606   0.239606     0.239606
RUGBYUNTOUCH_Q14    0.239606   0.239606     0.239606
CLIMBWALL_R02       0.239606   0.239606     0.239606
MOTORCARRACE_U28    0.239606   0.239606     0.239606
MOTORCYCRACE_U27    0.239606   0.239606     0.239606
GYMNASTICSONLY_U24  0.239606   0.239606     0.239606
Activitie

# 5. Reshape into long format, combining three participation candidates

In [6]:
id_cols = ['serial', 'wt_final', 'wt_time', 'Age16plus', 'Age9', 'Disab3'] + \
          [c for c in df.columns if c.startswith('disty')]

activities = [c.replace('MEMS7GR_', '') for c in mems7gr_cols]

frames = []
for act in activities:
    sub = df[id_cols + [f'MEMS7GR_{act}', f'MONTHS_12_{act}', f'DAYS10P60GR_{act}']].copy()
    sub['activity'] = act
    sub = sub.rename(columns={
        f'MEMS7GR_{act}': 'MEMS7GR',
        f'MONTHS_12_{act}': 'MONTHS_12',
        f'DAYS10P60GR_{act}': 'DAYS10P60GR'
    })
    frames.append(sub)

long_df = pd.concat(frames, ignore_index=True)

print(long_df.shape)
output_path = r"C:\Users\Lenovo\Desktop\Dissertation\DATa\7.12-15_RQ3_data\year7_long_table.csv"
long_df.to_csv(output_path, index=False)


(2017375, 23)


## 6. File paths for all eight years
Only the 125-activity version is used. 


In [1]:
import os

root = r"C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20"

year_files = {
    1: os.path.join(root, "Yifeng Mao", "data", "active_lives_1516_london_125.csv"),
    2: os.path.join(root, "Yifeng Mao", "data", "active_lives_1617_london_125.csv"),
    3: os.path.join(root, "Siyan Xin", "2017~2018", "2017_data_125_activities.csv"),
    4: os.path.join(root, "Siyan Xin", "2018~2019", "2018_data_125_activities.csv"),
    5: os.path.join(root, "Shuhan Zhao", "docs", "1920_london32_stable125.csv"),
    6: os.path.join(root, "Shuhan Zhao", "docs", "2021_london32_stable125.csv"),
    7: os.path.join(root, "Jingyi Hua", "data", "processed", "year7_125activities.csv"),
    8: os.path.join(root, "Jingyi Hua", "data", "processed", "year8_125activities.csv"),
}

for year_number, file_path in year_files.items():
    print(year_number, os.path.exists(file_path), file_path)

1 True C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Yifeng Mao\data\active_lives_1516_london_125.csv
2 True C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Yifeng Mao\data\active_lives_1617_london_125.csv
3 True C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Siyan Xin\2017~2018\2017_data_125_activities.csv
4 True C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Siyan Xin\2018~2019\2018_data_125_activities.csv
5 True C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Shuhan Zhao\docs\1920_london32_stable125.csv
6 True C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Shuhan Zhao\docs\2021_london32_stable125.csv
7 True C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\data\processed\year7_125activities.csv
8 True C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\data\processed\year8_125activities.csv


In [12]:
def normalize_columns(df):
    rename_map = {}
    for col in df.columns:
        if col.lower() == 'age16plus':
            rename_map[col] = 'Age16plus'
    return df.rename(columns=rename_map)

## 7. Check that each file has the columns this pipeline needs

In [7]:
required_base_cols = ['Age16plus', 'Age9', 'Disab3', 'wt_time'] + [f'disty{i}_POP' for i in range(1, 14)]

def check_columns(path, year_number):
    cols = pd.read_csv(path, nrows=0).columns
    missing_base = [c for c in required_base_cols if c not in cols]
    has_mems7gr = any(c.startswith('MEMS7GR_') for c in cols)
    has_days = any(c.startswith('DAYS10P60GR_') for c in cols)
    has_months = any(c.startswith('MONTHS_12_') for c in cols)
    print(f"Year {year_number}: missing base columns = {missing_base}, "
          f"has MEMS7GR = {has_mems7gr}, has DAYS10P60GR = {has_days}, has MONTHS_12 = {has_months}")

for year_number, file_path in year_files.items():
    check_columns(file_path, year_number)

Year 1: missing base columns = ['Age16plus'], has MEMS7GR = True, has DAYS10P60GR = True, has MONTHS_12 = True
Year 2: missing base columns = [], has MEMS7GR = True, has DAYS10P60GR = True, has MONTHS_12 = True
Year 3: missing base columns = [], has MEMS7GR = True, has DAYS10P60GR = True, has MONTHS_12 = True
Year 4: missing base columns = [], has MEMS7GR = True, has DAYS10P60GR = True, has MONTHS_12 = True
Year 5: missing base columns = [], has MEMS7GR = True, has DAYS10P60GR = True, has MONTHS_12 = True
Year 6: missing base columns = [], has MEMS7GR = True, has DAYS10P60GR = True, has MONTHS_12 = True
Year 7: missing base columns = [], has MEMS7GR = True, has DAYS10P60GR = True, has MONTHS_12 = True
Year 8: missing base columns = [], has MEMS7GR = True, has DAYS10P60GR = True, has MONTHS_12 = True


## 8. Weighted participation rate helper

In [8]:
import numpy as np

def weighted_rate(sub, value_col, weight_col):
    valid = sub.dropna(subset=[value_col, weight_col])
    n = len(valid)
    if n == 0:
        return np.nan, 0, 0.0
    w = valid[weight_col]
    part = (valid[value_col] > 0).astype(int)
    weighted_n = w.sum()
    rate = (part * w).sum() / weighted_n if weighted_n > 0 else np.nan
    return rate, n, weighted_n


## 9. Aggregation function for a single year

In [22]:
def aggregate_year(path, year_number):
    year_df = pd.read_csv(path)
    year_df = normalize_columns(year_df)
    year_df = year_df[(year_df['Age16plus'] == 1) & (year_df['Age9'].between(2, 9))].copy()

    mems7gr_cols_y = [c for c in year_df.columns if c.startswith('MEMS7GR_') and c != 'MEMS7GR_ALL']
    days_cols_y = [c for c in year_df.columns if c.startswith('DAYS10P60GR_')]
    months_cols_y = [c for c in year_df.columns if c.startswith('MONTHS_12_')]

    mems7gr_suffixes = set(c.replace('MEMS7GR_', '') for c in mems7gr_cols_y)
    days_suffixes = set(c.replace('DAYS10P60GR_', '') for c in days_cols_y)
    months_suffixes = set(c.replace('MONTHS_12_', '') for c in months_cols_y)

    activities_y = sorted(mems7gr_suffixes & days_suffixes & months_suffixes)

    excluded = (mems7gr_suffixes | days_suffixes | months_suffixes) - set(activities_y)
    if excluded:
        print(f"Year {year_number}: excluding {len(excluded)} activities due to inconsistent naming: {sorted(excluded)}")

    disty_cols_y = [f'disty{i}_POP' for i in range(1, 14)]

    records = []

    for act in activities_y:
        cols_needed = [f'MEMS7GR_{act}', f'DAYS10P60GR_{act}', f'MONTHS_12_{act}', 'wt_time', 'Age9']
        sub_full = year_df[cols_needed + disty_cols_y + ['Disab3']].copy()
        sub_full = sub_full.rename(columns={
            f'MEMS7GR_{act}': 'MEMS7GR',
            f'DAYS10P60GR_{act}': 'DAYS10P60GR',
            f'MONTHS_12_{act}': 'MONTHS_12'
        })

        for age_val, age_sub in sub_full.groupby('Age9'):

            r1, n1, wn1 = weighted_rate(age_sub, 'MEMS7GR', 'wt_time')
            r2, n2, wn2 = weighted_rate(age_sub, 'DAYS10P60GR', 'wt_time')
            r3, n3, wn3 = weighted_rate(age_sub, 'MONTHS_12', 'wt_time')
            records.append([year_number, age_val, act, 'total', np.nan, r1, r2, r3, n1, wn1])

            for dv, dsub in age_sub[age_sub['Disab3'].isin([1, 2, 3])].groupby('Disab3'):
                r1, n1, wn1 = weighted_rate(dsub, 'MEMS7GR', 'wt_time')
                r2, n2, wn2 = weighted_rate(dsub, 'DAYS10P60GR', 'wt_time')
                r3, n3, wn3 = weighted_rate(dsub, 'MONTHS_12', 'wt_time')
                records.append([year_number, age_val, act, 'Disab3', dv, r1, r2, r3, n1, wn1])

            for dcol in disty_cols_y:
                valid_sub = age_sub[age_sub[dcol].isin([0, 1])]
                for dv, dsub in valid_sub.groupby(dcol):
                    r1, n1, wn1 = weighted_rate(dsub, 'MEMS7GR', 'wt_time')
                    r2, n2, wn2 = weighted_rate(dsub, 'DAYS10P60GR', 'wt_time')
                    r3, n3, wn3 = weighted_rate(dsub, 'MONTHS_12', 'wt_time')
                    records.append([year_number, age_val, act, dcol.replace('_POP', ''), dv, r1, r2, r3, n1, wn1])

        for dv, dsub in sub_full[sub_full['Disab3'].isin([1, 2, 3])].groupby('Disab3'):
            r1, n1, wn1 = weighted_rate(dsub, 'MEMS7GR', 'wt_time')
            r2, n2, wn2 = weighted_rate(dsub, 'DAYS10P60GR', 'wt_time')
            r3, n3, wn3 = weighted_rate(dsub, 'MONTHS_12', 'wt_time')
            records.append([year_number, 'ALL', act, 'Disab3', dv, r1, r2, r3, n1, wn1])

        for dcol in disty_cols_y:
            valid_sub = sub_full[sub_full[dcol].isin([0, 1])]
            for dv, dsub in valid_sub.groupby(dcol):
                r1, n1, wn1 = weighted_rate(dsub, 'MEMS7GR', 'wt_time')
                r2, n2, wn2 = weighted_rate(dsub, 'DAYS10P60GR', 'wt_time')
                r3, n3, wn3 = weighted_rate(dsub, 'MONTHS_12', 'wt_time')
                records.append([year_number, 'ALL', act, dcol.replace('_POP', ''), dv, r1, r2, r3, n1, wn1])

    year_result = pd.DataFrame(records, columns=[
        'year', 'age_group', 'activity', 'group_type', 'group_value',
        'participation_MEMS7GR', 'participation_DAYS10P60GR', 'participation_MONTHS_12',
        'sample_n', 'weighted_n'
    ])
    return year_result

## 10. Run aggregation for all eight years and combine

In [23]:
all_years_results = []
problem_years = {}

for year_number in sorted(year_files.keys()):
    print('Processing year', year_number)
    try:
        year_result = aggregate_year(year_files[year_number], year_number)
        all_years_results.append(year_result)
    except KeyError as e:
        temp_df = pd.read_csv(year_files[year_number], nrows=5)
        print(f'Year {year_number} is missing column {e}')
        print('Columns containing age, disab, or wt in this file:')
        print([col for col in temp_df.columns
               if 'age' in col.lower() or 'disab' in col.lower() or 'wt' in col.lower()])
        problem_years[year_number] = list(temp_df.columns)

if all_years_results:
    final_table = pd.concat(all_years_results, ignore_index=True)
    print(final_table.shape)

print('Years with problems:', list(problem_years.keys()))

Processing year 1
Processing year 2
Year 2: excluding 5 activities due to inconsistent naming: ['INOUT_HOME_BOXINGTRAD_S01', 'INOUT_HOME_DANCEART_B05', 'INOUT_HOME_GARDENTRAMP_L14', 'INOUT_HOME_GYMNASTICSONLY_U24', 'INOUT_HOME_TABLETENNIS_G03']
Processing year 3
Processing year 4
Processing year 5
Processing year 6
Processing year 7
Processing year 8
(268731, 10)
Years with problems: []


## 11. Save the final aggregated table

In [24]:
output_dir = r"C:\Users\Lenovo\Desktop\Dissertation\DATa\7.12-15_RQ3_data"
os.makedirs(output_dir, exist_ok=True)

final_output_path = os.path.join(output_dir, "RQ3_aggregated_participation_all_years.csv")
final_table.to_csv(final_output_path, index=False)
print('Saved to', final_output_path)

Saved to C:\Users\Lenovo\Desktop\Dissertation\DATa\7.12-15_RQ3_data\RQ3_aggregated_participation_all_years.csv
